# 🔴 [SOLUTION] IBM Quantum Safe Advanced Lab: Stateful Crypto-Agile Security Engine

## 🎯 Learning Objectives
1. Understand the architectural design patterns of **Crypto-Agility** in enterprise web services.
2. Build a stateful, dynamic Security Engine capable of switching between Classical, Hybrid, and Pure Post-Quantum cipher mapping on-the-fly.
3. Simulate an API gateway intercepting traffic and routing with zero-downtime policy hot-swapping.

---
## 🧱 Task 1: Complete the Stateful Agile Security Engine
In a crypto-agile setup, the application never references raw ciphers directly. Instead, it queries a stateful engine to retrieve active ciphers.

### State Mappings to Implement:
1. **CLASSICAL Mode**:
   - Key Exchange: `'ECDH-P256'`
   - Signature: `'RSA-2048'`
   - Symmetric: `'AES-128-GCM'`
2. **HYBRID Mode**:
   - Key Exchange: `'X25519-MLKEM-768'`
   - Signature: `'RSA-MLDSA-65'`
   - Symmetric: `'AES-256-GCM'`
3. **POST_QUANTUM Mode**:
   - Key Exchange: `'ML-KEM-768'`
   - Signature: `'ML-DSA-65'`
   - Symmetric: `'AES-256-GCM'`

In [ ]:
from typing import Dict, Any

class AgileSecurityEngine:
    """
    Stateful engine enforcing cryptographic policies dynamically:
    Supported modes: 'CLASSICAL', 'HYBRID', 'POST_QUANTUM'
    """
    def __init__(self, initial_policy: str = "CLASSICAL"):
        self.policy = initial_policy

    def set_policy(self, new_policy: str) -> None:
        if new_policy not in ["CLASSICAL", "HYBRID", "POST_QUANTUM"]:
            raise ValueError("Invalid policy mode")
        self.policy = new_policy

    def get_cipher_suite(self) -> Dict[str, str]:
        """Returns the active cipher suite mapping based on the current policy."""
        if self.policy == "CLASSICAL":
            return {
                "key_exchange": "ECDH-P256",
                "signature": "RSA-2048",
                "symmetric": "AES-128-GCM"
            }
        elif self.policy == "HYBRID":
            return {
                "key_exchange": "X25519-MLKEM-768",
                "signature": "RSA-MLDSA-65",
                "symmetric": "AES-256-GCM"
            }
        elif self.policy == "POST_QUANTUM":
            return {
                "key_exchange": "ML-KEM-768",
                "signature": "ML-DSA-65",
                "symmetric": "AES-256-GCM"
            }
        else:
            raise ValueError("Unknown policy state")

---
## 🧱 Task 2: Build the Dynamic API Gateway Proxy
Your second task is to build an API gateway function that inspects client headers, determines if client capabilities require a specific post-quantum standard, hot-swaps policies inside your stateful engine, and proxies the connection securely without service interruption.

In [ ]:
def handle_api_request(engine: AgileSecurityEngine, http_headers: Dict[str, str], request_path: str) -> Dict[str, Any]:
    """
    Inspects request, updates security engine states dynamically on-the-fly,
    and returns a connection proxy envelope detailing the active ciphers used.
    """
    client_cap = http_headers.get("X-Client-Cap", "")
    
    if request_path == "/secure/quantum-hq" or "ML-KEM" in client_cap and "Hybrid" not in client_cap:
        engine.set_policy("POST_QUANTUM")
    elif "Hybrid-Ready" in client_cap:
        engine.set_policy("HYBRID")
    else:
        engine.set_policy("CLASSICAL")
    
    active_ciphers = engine.get_cipher_suite()
    
    return {
        "request_path": request_path,
        "negotiated_policy": engine.policy,
        "active_ciphers": active_ciphers,
        "status": "Connection Established - Zero Downtime State Swapped"
    }

---
## 🧪 Task 3: Test Dynamic Agility Hot-Swapping
Run the client requests simulation below to see policy changes in real-time.

In [ ]:
engine = AgileSecurityEngine("CLASSICAL")

print("--- Client Request 1: Legacy Browser ---")
req1 = handle_api_request(engine, {"X-Client-Cap": "RSA, AES-128"}, "/public/index.html")
print(f"Negotiated Mode: {req1['negotiated_policy']}")
print(f"Ciphers in Use:  {req1['active_ciphers']}\n")

print("--- Client Request 2: Financial API (Hybrid-Ready) ---")
req2 = handle_api_request(engine, {"X-Client-Cap": "Hybrid-Ready, ML-KEM"}, "/finance/transfer")
print(f"Negotiated Mode: {req2['negotiated_policy']}")
print(f"Ciphers in Use:  {req2['active_ciphers']}\n")

print("--- Client Request 3: Quantum Secure High-Security End Point ---")
req3 = handle_api_request(engine, {}, "/secure/quantum-hq")
print(f"Negotiated Mode: {req3['negotiated_policy']}")
print(f"Ciphers in Use:  {req3['active_ciphers']}\n")